### Day 34 AM Session | Week 6 Clustering: K-Means, DBSCAN & Hierarchical

---
| Part | Content |
|---|---|
| Part A | K-Means & DBSCAN on Iris — ARI, NMI, Visualization |
| Part B | Hierarchical Clustering + Dendrogram |
| Part C | Interview Questions |
| Part D | Clustering Analogy Analysis |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, confusion_matrix, accuracy_score
)
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.optimize import linear_sum_assignment
import time


---
## Part A K-Means & DBSCAN on Iris (Unsupervised)

In [ ]:
def load_and_scale_iris():
    iris = load_iris()
    X, y = iris.data, iris.target
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y, list(iris.target_names)


def run_kmeans(X, k, random_state=42):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=random_state)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    return km, labels, sil


def align_cluster_labels(true_labels, pred_labels):
    n = max(len(np.unique(pred_labels)), len(np.unique(true_labels)))
    cost = np.zeros((n, n), dtype=int)
    for t, p in zip(true_labels, pred_labels):
        cost[p, t] += 1
    row_ind, col_ind = linear_sum_assignment(-cost)
    mapping = {r: c for r, c in zip(row_ind, col_ind)}
    return np.array([mapping[p] for p in pred_labels])


def cluster_comparison_table(true_labels, pred_labels, target_names):
    aligned = align_cluster_labels(true_labels, pred_labels)
    cm = confusion_matrix(true_labels, aligned)
    df = pd.DataFrame(
        cm,
        index=[f"True: {n}" for n in target_names],
        columns=[f"Cluster {i}" for i in range(cm.shape[1])]
    )
    return df, aligned


def compute_clustering_metrics(true_labels, pred_labels):
    ari = adjusted_rand_score(true_labels, pred_labels)
    nmi = normalized_mutual_info_score(true_labels, pred_labels)
    return ari, nmi


X_iris, y_iris, target_names = load_and_scale_iris()
print(f"Shape  : {X_iris.shape}")
print(f"Classes: {target_names}")


In [ ]:
km_model, km_labels, km_sil = run_kmeans(X_iris, k=3)
ari_km, nmi_km = compute_clustering_metrics(y_iris, km_labels)

print("K-Means (K=3) Results")
print(f"  Silhouette Score : {km_sil:.4f}")
print(f"  ARI              : {ari_km:.4f}")
print(f"  NMI              : {nmi_km:.4f}")

comp_df, aligned_km = cluster_comparison_table(y_iris, km_labels, target_names)
print("\nCluster vs True Label Comparison:")
print(comp_df)
print(f"\nAligned Accuracy : {accuracy_score(y_iris, aligned_km):.4f}")


In [ ]:
def plot_elbow_and_silhouette(X, k_range=range(2, 11), figsize=(12, 4)):
    inertias    = []
    sil_scores  = []
    for k in k_range:
        km  = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
        lbl = km.fit_predict(X)
        inertias.append(km.inertia_)
        sil_scores.append(silhouette_score(X, lbl))

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', linewidth=2)
    axes[0].set_xlabel('K')
    axes[0].set_ylabel('Inertia')
    axes[0].set_title('Elbow Method')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(list(k_range), sil_scores, 's-', color='darkorange', linewidth=2)
    axes[1].axhline(0.25, color='red', linestyle='--', alpha=0.5, label='0.25 threshold')
    axes[1].set_xlabel('K')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Silhouette vs K')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Optimal K Selection — Iris', fontsize=12)
    plt.tight_layout()
    plt.show()
    return list(k_range), inertias, sil_scores


k_vals, inertias, sil_scores = plot_elbow_and_silhouette(X_iris)
best_k = k_vals[np.argmax(sil_scores)]
print(f"Best K by Silhouette : {best_k}  (score={max(sil_scores):.4f})")


In [ ]:
def tune_dbscan(X, eps_range, min_samples_range):
    results = []
    for eps in eps_range:
        for ms in min_samples_range:
            db = DBSCAN(eps=eps, min_samples=ms)
            labels = db.fit_predict(X)
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise    = (labels == -1).sum()
            if n_clusters >= 2:
                mask = labels != -1
                sil  = silhouette_score(X[mask], labels[mask]) if mask.sum() > n_clusters else -1
                results.append({
                    'eps': round(eps, 2), 'min_samples': ms,
                    'n_clusters': n_clusters, 'n_noise': n_noise,
                    'silhouette': round(sil, 4)
                })
    return pd.DataFrame(results).sort_values('silhouette', ascending=False)


def run_dbscan(X, eps, min_samples):
    db = DBSCAN(eps=eps, min_samples=min_samples)
    labels = db.fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    return db, labels, n_clusters, n_noise


eps_vals = np.arange(0.3, 1.3, 0.1).round(1)
ms_vals  = [3, 5, 7, 10]
tuning_df = tune_dbscan(X_iris, eps_vals, ms_vals)
print("DBSCAN Tuning (top 8):")
print(tuning_df.head(8).to_string(index=False))


In [ ]:
best_row = tuning_df.iloc[0]
db_model, db_labels, n_db_clusters, n_noise = run_dbscan(
    X_iris, eps=best_row['eps'], min_samples=int(best_row['min_samples'])
)

ari_db, nmi_db = compute_clustering_metrics(y_iris, db_labels)
print(f"DBSCAN (eps={best_row['eps']}, min_samples={int(best_row['min_samples'])})")
print(f"  Clusters found   : {n_db_clusters}")
print(f"  Noise points     : {n_noise}")
print(f"  ARI              : {ari_db:.4f}")
print(f"  NMI              : {nmi_db:.4f}")


In [ ]:
def plot_cluster_comparison(X, y_true, y_km, y_db, target_names, figsize=(15, 5)):
    pca2 = PCA(n_components=2)
    X_2d = pca2.fit_transform(X)
    var  = pca2.explained_variance_ratio_

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    cmap = plt.cm.Set1
    configs = [
        (y_true, "True Labels",   target_names),
        (y_km,   "K-Means (K=3)", [f"Cluster {i}" for i in range(3)]),
        (y_db,   "DBSCAN",        None)
    ]

    for ax, (labels, title, names) in zip(axes, configs):
        unique = sorted(set(labels))
        colors = cmap(np.linspace(0, 0.8, len(unique)))
        for color, lbl in zip(colors, unique):
            mask = labels == lbl
            if names and lbl >= 0:
                name = names[lbl]
            elif lbl == -1:
                name = "Noise"
            else:
                name = f"Cluster {lbl}"
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[color],
                       label=name, s=45, alpha=0.85, edgecolors='k', linewidths=0.3)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel(f"PC1 ({var[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({var[1]*100:.1f}%)")
        ax.legend(fontsize=8)

    plt.suptitle("Iris: True Labels vs K-Means vs DBSCAN (PCA 2D)", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


plot_cluster_comparison(X_iris, y_iris, aligned_km, db_labels, target_names)

print("Interpretation:")
print("  Setosa is perfectly recovered — it is linearly separable in feature space.")
print("  Versicolor and Virginica overlap — their confusion reflects genuine similarity.")
print("  When unsupervised clustering matches true labels, the features carry real")
print("  discriminative signal about the underlying natural groups.")


---
## Part B Hierarchical Clustering + Dendrogram

In [ ]:
def run_hierarchical(X, n_clusters=3, linkage_method='ward'):
    hc = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage_method)
    labels = hc.fit_predict(X)
    sil = silhouette_score(X, labels)
    return hc, labels, sil


def plot_dendrogram(X, method='ward', truncate_p=30, figsize=(12, 5)):
    Z = linkage(X, method=method)
    fig, ax = plt.subplots(figsize=figsize)
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=truncate_p,
               leaf_rotation=45, leaf_font_size=9, show_contracted=True)
    ax.set_title(f"Dendrogram — {method.capitalize()} Linkage", fontsize=12)
    ax.set_xlabel("Sample / Cluster Size")
    ax.set_ylabel("Distance")
    ax.axhline(y=6.5, color='red', linestyle='--', linewidth=1.2,
               label='Cut → 3 clusters')
    ax.legend()
    plt.tight_layout()
    plt.show()


hc_model, hc_labels, hc_sil = run_hierarchical(X_iris, n_clusters=3)
ari_hc, nmi_hc = compute_clustering_metrics(y_iris, hc_labels)

print("Hierarchical Clustering (Ward Linkage, n=3)")
print(f"  Silhouette : {hc_sil:.4f}")
print(f"  ARI        : {ari_hc:.4f}")
print(f"  NMI        : {nmi_hc:.4f}")

plot_dendrogram(X_iris, method='ward')


In [ ]:
def safe_db_sil(X, labels):
    mask = labels != -1
    if mask.sum() > len(set(labels[mask])):
        return silhouette_score(X[mask], labels[mask])
    return 0.0


summary = pd.DataFrame({
    'Method':     ['K-Means (K=3)', 'DBSCAN', 'Hierarchical (Ward)'],
    'ARI':        [ari_km,          ari_db,   ari_hc],
    'NMI':        [nmi_km,          nmi_db,   nmi_hc],
    'Silhouette': [km_sil,          safe_db_sil(X_iris, db_labels), hc_sil]
}).round(4).sort_values('ARI', ascending=False).reset_index(drop=True)
summary.index += 1

print("Three-Method Comparison:")
print(summary.to_string())


---
## Part C Interview Ready

### Q1 Conceptual: K-Means as a Greedy Algorithm

K-Means is **greedy** because at each assignment step it makes the locally optimal decision (nearest centroid) without reconsidering it in future iterations. There is no backtracking.

**Can it get stuck in local minima? Yes.** The algorithm is guaranteed to converge, but convergence is only to a local minimum of the inertia objective. Two different random starts can produce two different local optima — neither is globally optimal.

**K-Means++ Initialization** reduces this risk by choosing initial centroids with probability proportional to their squared distance from already-selected centroids:

```
P(x is chosen next) ∝ D(x)²
```

This ensures centroids are spread across the data at the start, substantially reducing the probability of a poor local minimum. In practice, K-Means++ converges in fewer iterations and achieves lower inertia than random initialization.

| Property | Random Init | K-Means++ |
|---|---|---|
| Initialization time | O(k) | O(nk) |
| Quality | Variable | Consistently near-optimal |
| Convergence speed | Slow | Fast |
| sklearn default | No | Yes |


In [ ]:
def kmeans_from_scratch(X, k, max_iter=100, tol=1e-4, random_state=42):
    if k < 1 or k > len(X):
        raise ValueError(f"k must be between 1 and {len(X)}, got {k}")
    if max_iter < 1:
        raise ValueError("max_iter must be at least 1")

    rng = np.random.RandomState(random_state)
    centroids = X[rng.choice(len(X), k, replace=False)].astype(float)
    labels = np.zeros(len(X), dtype=int)

    for _ in range(max_iter):
        diff      = X[:, np.newaxis, :] - centroids[np.newaxis, :, :]
        distances = np.sqrt((diff ** 2).sum(axis=2))
        new_labels = distances.argmin(axis=1)

        new_centroids = np.array([
            X[new_labels == c].mean(axis=0) if (new_labels == c).any()
            else centroids[c]
            for c in range(k)
        ])

        shift = np.sqrt(((new_centroids - centroids) ** 2).sum(axis=1)).max()
        centroids, labels = new_centroids, new_labels

        if shift < tol:
            break

    inertia = sum(
        ((X[labels == c] - centroids[c]) ** 2).sum()
        for c in range(k) if (labels == c).any()
    )
    return labels, centroids, inertia


scratch_labels, scratch_centroids, scratch_inertia = kmeans_from_scratch(X_iris, k=3)
scratch_ari, _ = compute_clustering_metrics(y_iris, scratch_labels)
sklearn_ari     = ari_km

print(f"K-Means from Scratch  ARI : {scratch_ari:.4f}")
print(f"sklearn K-Means       ARI : {sklearn_ari:.4f}")
print(f"Inertia               : {scratch_inertia:.4f}")
print(f"Centroid shape        : {scratch_centroids.shape}")


*italicised text*### Q3  Analyze: Silhouette Score = 0.25 with K=5

**Is 0.25 good?**

| Range | Interpretation |
|---|---|
| 0.71–1.00 | Strong structure |
| 0.51–0.70 | Reasonable structure |
| 0.26–0.50 | Weak structure |
| ≤ 0.25 | No substantial structure found |

0.25 is at the lower boundary — barely acceptable. Investigate these possibilities:

**1. Wrong K:** The true natural grouping may not be 5. Re-plot the elbow and silhouette across K=2 to 10 — the peak silhouette may be at a different K.

**2. Non-spherical clusters:** K-Means assumes convex, equally-sized clusters. Visualize in 2D with PCA. If clusters are elongated or ring-shaped, switch to DBSCAN or Gaussian Mixture Models.

**3. Curse of dimensionality:** In high dimensions, Euclidean distances become uniform across all pairs. Apply PCA to 10–20 dimensions before clustering.

**4. No real cluster structure:** The data may be unimodal with no natural groupings. A silhouette peak at K=2 that is still low confirms this.

**5. Unscaled features:** Confirm StandardScaler was applied — unequal feature scales create artificially elongated distance landscapes.


---
## Part D Clustering Analogy: Sorting Fruits

### K-Means Sorting by Average Position
You place K baskets on a table, throw each fruit into the nearest basket, then slide each basket to the average position of its fruits. Repeat until baskets stop moving.

**Fails when:** You try to sort bananas this way. Bananas are elongated — the average banana position is in the middle of the fruit, which is not where any banana is. K-Means would split one banana cluster into two spherical halves.

### DBSCAN Sorting by Crowd Density
Walk through the fruit pile. If you find at least `min_samples` fruits within arm's reach (`eps`), mark them as a cluster and expand outward. Lone fruits far from any group get a "noise" label.

**Fails when:** Fruits are uniformly distributed with no density gaps (e.g., a perfectly shuffled mix with equal spacing). DBSCAN cannot find a meaningful boundary and either merges everything or labels everything as noise.

### Hierarchical Building a Family Tree
Start with every fruit as its own group. Repeatedly merge the two most similar groups. The dendrogram shows the full merge history — cut at any height to get K clusters.

**Fails when:** The fruit pile has millions of items. The pairwise distance matrix requires O(n²) memory — infeasible at scale.

### Verified Accuracy Assessment
The analogy is accurate for intuition. One real nuance DBSCAN's `eps` misses: two visually identical fruits with very different weights (dense mango vs hollow papaya) would be far apart in feature space even if physically adjacent — the analogy assumes only spatial proximity.
